# Perf metrics

Последний прогон `run-all`: wall-time и стадии из `data/qa/command_timing.jsonl`.

In [ ]:
from __future__ import annotations

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

from mobile.command_timing import DEFAULT_METRICS_PATH, infer_latest_run_id, load_command_metrics_df
from mobile.project_paths import PROJECT_ROOT

ROOT = PROJECT_ROOT
METRICS = ROOT / DEFAULT_METRICS_PATH

In [ ]:
raw = load_command_metrics_df(METRICS)
if raw.empty:
    print('Нет записей в', METRICS, '— выполните run-all или отдельные build под command_run_scope')
else:
    run_id = infer_latest_run_id(raw)
    latest = raw[raw['run_id'] == run_id].copy() if run_id and 'run_id' in raw.columns else raw
    print('run_id:', run_id, '| commands:', len(latest))
    display(latest.sort_values('elapsed_total_sec', ascending=False)[['command', 'elapsed_total_sec']].head(40))

In [ ]:
if raw.empty:
    pass
else:
    build = latest[latest['command'].str.startswith('build-')].sort_values('elapsed_total_sec', ascending=True)
    other = latest[~latest['command'].str.startswith('build-')].sort_values('elapsed_total_sec', ascending=True)
    fig, axes = plt.subplots(1, 2, figsize=(16, max(5, 0.3 * max(len(build), len(other), 1))))
    for ax, part, title in ((axes[0], build, 'build'), (axes[1], other, 'other')):
        if part.empty:
            ax.set_title(title + ' (нет данных)')
            continue
        ax.barh(part['command'], part['elapsed_total_sec'], color='#2a6fdb')
        ax.set_title(f'Последний прогон: {title}')
        ax.grid(axis='x', alpha=0.3)
    plt.tight_layout()
    plt.show()
    if 'run-all' in latest['command'].values:
        wall = float(latest.loc[latest['command'] == 'run-all', 'elapsed_total_sec'].iloc[-1])
        print(f'run-all wall: {wall:.1f}s')
    stage_cols = [c for c in latest.columns if c.endswith('_sec') and c != 'elapsed_total_sec']
    rows = []
    for _, row in latest.iterrows():
        for col in stage_cols:
            value = row.get(col)
            if pd.notna(value) and float(value) > 0:
                rows.append({'command': row['command'], 'stage': col, 'sec': float(value)})
    if rows:
        stages = pd.DataFrame(rows).sort_values('sec', ascending=False).head(20)
        fig, ax = plt.subplots(figsize=(11, 6))
        ax.barh(stages['command'] + ' / ' + stages['stage'], stages['sec'], color='#27ae60')
        ax.set_title('Top stages (latest run)')
        plt.tight_layout()
        plt.show()